In [1]:
import os, json
import pandas as pd
import numpy as np



In [2]:
np.random.seed(2020)



In [3]:
pred_cols = ["reactivity", "deg_Mg_pH10", "deg_Mg_50C", "deg_pH10", "deg_50C"]




In [4]:
def pandas_list_to_array(df):
    """
    Convert a DataFrame whose cells contain equal‑length lists into a 3‑D numpy array.
    Input shape  (samples, cols) where each entry is a list of length L.
    Output shape (samples, L, cols)
    """
    arr = np.array(df.values.tolist())
    # transpose to (samples, L, cols)
    return np.transpose(arr, (0, 2, 1))




In [5]:
data_dir = "/kaggle/input/stanford-covid-vaccine/"
train = pd.read_json(data_dir + "train.json", lines=True)
test = pd.read_json(data_dir + "test.json", lines=True)
sample_df = pd.read_csv(data_dir + "sample_submission.csv")



In [6]:
# Keep only reasonably high‑quality training rows (same filter as original)
train = train.query("signal_to_noise >= 1")



In [7]:
# Compute per‑position mean for each of the 5 target columns (using the first 68 positions)
# The training label arrays have shape (samples, 68, 5)
train_labels = pandas_list_to_array(train[pred_cols])  # (samples, 68, 5)

# Mean over samples → (68, 5)
pos_means = np.mean(train_labels, axis=0)  # shape (68, 5)



In [8]:
# Prepare prediction arrays for the public and private test sets.
# Public test set always has seq_length == 107; private (if any) would be 130.
public_df = test.query("seq_length == 107")
private_df = test.query("seq_length == 130")  # will be empty for this dataset


def build_predictions(df, seq_len):
    """Return an array (samples, seq_len, 5) filled with position‑wise means.
    For positions beyond 68 we fill with zeros (they are not scored)."""
    n = len(df)
    preds = np.zeros((n, seq_len, 5), dtype=np.float32)
    # copy the means for the first 68 positions
    preds[:, :68, :] = pos_means
    return preds


public_preds = build_predictions(public_df, seq_len=107)
private_preds = build_predictions(private_df, seq_len=130)  # empty if df is empty



In [9]:
# Assemble the predictions into the required long‑format DataFrame
preds_ls = []
for df, preds in [(public_df, public_preds), (private_df, private_preds)]:
    for i, uid in enumerate(df["id"].values):
        single_pred = preds[i]  # (seq_len, 5)
        single_df = pd.DataFrame(single_pred, columns=pred_cols)
        single_df["id_seqpos"] = [f"{uid}_{pos}" for pos in range(single_df.shape[0])]
        preds_ls.append(single_df)

preds_df = pd.concat(preds_ls, ignore_index=True)



In [10]:
# Merge with sample submission to guarantee correct ordering / missing rows
submission = sample_df[["id_seqpos"]].merge(preds_df, on="id_seqpos", how="left")

# Any missing predictions (should not happen) are filled with zeros
submission[pred_cols] = submission[pred_cols].fillna(0)

submission.to_csv("submission.csv", index=False)